# 04 · Evaluation

| Stage | Notebook |
|---|---|
| 1. Data quality & EDA | `01_eda.ipynb` |
| 2. Feature engineering | `02_feature_engineering.ipynb` |
| 3. Model training | `03_model_training.ipynb` |
| **4. Evaluation** | **this notebook** |

**Inputs:** the artefacts written by `python scripts/run_phase1.py`, the single source of truth for the reported numbers:
* `results/predictions.csv.gz`: uncalibrated scores of every model on every flight. Calibration, metrics, policies, drift and the bootstrap are **recomputed here** from these scores with the same `aerocast.pipeline` functions, and checked against the script's saved tables.
* Ablation, permutation importance, gain and coefficients need refitted models, so they are **loaded** from `results/`.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))  # make `aerocast` importable
import warnings; warnings.filterwarnings("ignore")
import io, json
import numpy as np, pandas as pd
from IPython.display import display
from aerocast import config, pipeline, plots
pd.set_option("display.precision", 4); pd.set_option("display.width", 160)
R = config.RESULTS_DIR
S = json.loads((R / "summary.json").read_text())

def same_as_saved(df, name):
    """Exact comparison with a saved CSV (round-trip float parsing)."""
    a = pd.read_csv(io.StringIO(df.to_csv(index=False)), float_precision="round_trip")
    b = pd.read_csv(R / name, float_precision="round_trip")
    pd.testing.assert_frame_equal(a, b)
    return f"OK: identical to results/{name}"

labels, raw_preds = pipeline.load_predictions()
pd.Series({s: len(y) for s, y in labels.y.items()}, name="rows").to_frame().T

,train,calib,test
rows,211684,55628,53991


## 1. Calibrator selection (fit on Sep → check on Oct)

Candidates: **raw** (no recalibration), **Platt** (logistic regression on the score's logit, the two-class analogue of Guo et al.'s temperature scaling), and **isotonic** regression.

Choosing a calibrator by its score on the data it was fitted on would always favour isotonic. Instead we do a **forward check inside the calibration window**: fit each candidate on September, measure ECE on October, then refit the winner on all of Sep–Oct. The test set plays no part.

In [2]:
final_preds, choice, calib_tbl = pipeline.calibrate(labels, raw_preds)
print(same_as_saved(calib_tbl, "calibrator_selection.csv"))
print("selected:", choice)
calib_tbl.pivot(index="calibrator", columns="model", values="check_ece")

OK: identical to results/calibrator_selection.csv
selected: {'logreg': 'platt', 'xgboost': 'raw'}


model,logreg,xgboost
calibrator,,
isotonic,0.0399,0.0607
platt,0.0398,0.0605
raw,0.0405,0.0269


* **XGBoost → raw**, clearly the best on October (ECE 0.027 vs 0.060).
* **LogReg → Platt**, but by a near-tie (0.0398 vs 0.0405). We'll see in §5 that this choice doesn't hold up on test. We report that rather than re-choosing after seeing test results.

## 2. Test metrics (Nov–Dec)

* **PR-AUC** (primary): ranking quality on the late class; the baseline is the base rate.
* **ROC-AUC**: ranking quality, unaffected by the base rate.
* **Brier** score and **ECE** (expected calibration error, 15 equal-width bins): probability quality.
* **Recall @ precision ≥ p**: the best recall achievable while at least p of the flagged flights are truly late. This is the brief's operational metric, measured at p ∈ {0.30, 0.40, 0.50}.

In [3]:
thresholds = pipeline.frozen_thresholds(labels, final_preds)
results = pipeline.metrics_table(labels, final_preds, thresholds)
print(same_as_saved(results, "metrics_all.csv"))
pipeline.headline_table(results)[pipeline.SHOW_COLS]

OK: identical to results/metrics_all.csv


,pr_auc,roc_auc,brier,ece,mean_pred,base_rate,recall@p0.30,recall@p0.40,recall@p0.50
model,,,,,,,,,
prior,0.2583,0.5000,0.1916,0.0066,0.2649,0.2583,0.0000,0.0000,0.0000
route_history,0.3398,0.6148,0.1875,0.0401,0.2227,0.2583,0.7910,0.0947,0.0028
logreg_cal,0.4054,0.6658,0.1840,0.0719,0.1865,0.2583,0.8687,0.4837,0.1590
xgboost_cal,0.3922,0.6637,0.1808,0.0358,0.2253,0.2583,0.8792,0.4335,0.1087


* Both learned models clearly beat the **prior** (PR-AUC 0.258) and the **route-history** heuristic (0.340): **+0.05–0.07 PR-AUC** and **+0.05 ROC-AUC** over the heuristic.
* At precision ≥ 0.40, the models recover **43–48%** of late flights. The heuristic recovers 9%.
* LogReg ranks slightly better, while XGBoost has better probabilities (lower Brier and ECE). Whether the ranking difference is real is tested next.

## 3. Is XGBoost better than LogReg? Day-block bootstrap

Flights on the same day share weather and air-traffic shocks, so they **aren't independent**. Resampling individual flights would make the confidence intervals too narrow. Instead we resample **whole days** of the test period (500 replicates) and recompute both models' metrics on each resample. The difference is computed on the same days for both models (a paired comparison).

In [4]:
boot = pipeline.bootstrap_compare(labels, final_preds)
assert json.loads(json.dumps(boot)) == S["bootstrap_test"], "bootstrap differs from summary.json"
pd.DataFrame({k: {"XGBoost 95% CI": np.round(v["xgboost_ci95"], 3), "LogReg 95% CI": np.round(v["logreg_ci95"], 3),
                  "XGB − LR 95% CI": np.round(v["diff_xgboost_minus_logreg_ci95"], 3),
                  "P(XGB not better)": v["p_xgboost_not_better"]} for k, v in boot.items()}).T

,XGBoost 95% CI,LogReg 95% CI,XGB − LR 95% CI,P(XGB not better)
pr_auc,"[0.336, 0.458]","[0.346, 0.467]","[-0.03, 0.008]",0.886
roc_auc,"[0.634, 0.695]","[0.633, 0.694]","[-0.018, 0.017]",0.602


**The difference is not significant.** The 95% CI for XGB − LR includes 0 for both PR-AUC and ROC-AUC. With hand-built day-ahead summary features, a boosted tree ensemble finds **no interaction structure** that an additive model misses.

This is the "gap" the brief asks us to identify. Manual route and congestion summaries already capture what trees can extract. Whether *learned* carrier × origin × destination × time embeddings can find more is the question for the deep-learning phase.

## 4. Precision–recall and ROC curves

`pipeline.evaluation_figures` builds every evaluation figure, using the same function the script calls. Each figure is saved with `plots.save`.

In [5]:
figs = pipeline.evaluation_figures(labels, final_preds,
                                   pd.read_csv(R / "monthly_calibration_drift.csv"),
                                   pd.read_csv(R / "xgb_gain_importance.csv", index_col=0)["gain"],
                                   pd.read_csv(R / "permutation_importance_groups.csv"),
                                   pd.read_csv(R / "ablation.csv"))
for name, fig in figs.items():
    plots.save(fig, name)
figs["pr_roc_test"]

<Figure size 850x360 with 2 Axes>

The two models' curves lie almost on top of each other, consistent with the bootstrap. Both are above the heuristic everywhere. Precision starts around 0.55–0.6 for the very highest-risk flights and falls toward the 0.26 base rate as recall approaches 1.

## 5. Reliability diagrams (test)

In [6]:
figs["reliability_logreg_test"]

<Figure size 440x480 with 2 Axes>

In [7]:
figs["reliability_xgboost_test"]

<Figure size 440x480 with 2 Axes>

In [8]:
results[results.split.isin(["test", "test_m11", "test_m12"]) & results.model.str.match(r"(logreg|xgboost)_(raw|platt|isotonic)$")] \
    .pivot_table(index="model", columns="split", values=["ece", "mean_pred"]).round(4)

ece                   mean_pred                  
split               test test_m11 test_m12      test test_m11 test_m12
model                                                                 
logreg_isotonic   0.0735   0.0197   0.1276    0.1861   0.1617   0.2104
logreg_platt      0.0719   0.0201   0.1244    0.1865   0.1617   0.2112
logreg_raw        0.0139   0.0399   0.0589    0.2501   0.2201   0.2802
xgboost_isotonic  0.0977   0.0434   0.1520    0.1608   0.1377   0.1838
xgboost_platt     0.0989   0.0449   0.1533    0.1595   0.1367   0.1821
xgboost_raw       0.0358   0.0138   0.0711    0.2253   0.1811   0.2694

**Recalibration made test calibration worse.** For LogReg, raw ECE is 0.014 and Platt is 0.072. The raw models sit close to the diagonal, while the recalibrated curves under-predict (most points lie above the diagonal).

The reason is that Platt and isotonic both assume the calibration data are **distributed like the deployment data**. Here Sep–Oct is the calmest window of the year (15.7% late) and December the worst (33.5%), so the calibrators learn to scale risk *down*, and then December happens. November, which resembles Sep–Oct (18%), is well calibrated after recalibration (LogReg Platt ECE 0.020). December is not (0.124).

Guo et al. (2017) study calibration on data drawn from the same distribution as training. Ovadia et al. (2019) show that calibration degrades under dataset shift, and this is a concrete example of that.

We also tested a *shift-aware* Platt variant (adding the trailing 28-day network rate as an input) during development. It made test ECE **worse** (0.104): within Sep–Oct the trailing rate falls as summer fades, a relationship that doesn't hold in December.

## 6. Monthly calibration drift

In [9]:
drift = pipeline.monthly_drift(labels, final_preds)
print(same_as_saved(drift, "monthly_calibration_drift.csv"))
figs["monthly_drift_logreg"]

OK: identical to results/monthly_calibration_drift.csv


<Figure size 650x320 with 1 Axes>

In [10]:
drift.pivot_table(index="month", columns="model", values=["observed", "predicted"]).round(3)

observed                                         predicted                                       
model logreg_cal logreg_platt xgboost_cal xgboost_platt logreg_cal logreg_platt xgboost_cal xgboost_platt
month                                                                                                    
1          0.238        0.238       0.238         0.238      0.161        0.161       0.229         0.162
2          0.238        0.238       0.238         0.238      0.188        0.188       0.253         0.174
3          0.236        0.236       0.236         0.236      0.184        0.184       0.246         0.170
4          0.288        0.288       0.288         0.288      0.197        0.197       0.269         0.182
5          0.227        0.227       0.227         0.227      0.189        0.189       0.234         0.164
6          0.315        0.315       0.315         0.315      0.212        0.212       0.282         0.189
7          0.322        0.322       0.322         0.322      0.246        0.246       0.332         0.215
8          0.246        0.246       0.246         0.246      0.204        0.204       0.262         0.178
9          0.137        0.137       0.137         0.137      0.157        0.157       0.243         0.168
10         0.176        0.176       0.176         0.176      0.157        0.157       0.199         0.146
11         0.181        0.181       0.181         0.181      0.162        0.162       0.181         0.137
12         0.335        0.335       0.335         0.335      0.211        0.211       0.269         0.182

The predictions follow the **shape** of the season (higher in summer, a dip in autumn, rising in December) but **compress it toward the middle**. December's 33.5% is above anything the model saw in training, so its risk estimates can't reach that level. More than one year of data, or recalibrating on a rolling recent window, would address this.

## 7. Operating policies

The brief asks for recall at an operational precision. There are two ways to turn scores into actions:
* **Frozen threshold**: choose, on Sep–Oct, the lowest threshold that reaches precision ≥ 0.40, and keep it fixed for Nov–Dec.
* **Daily capacity** (top 10%): each day, flag the 10% highest-risk flights, as an operations desk with fixed review capacity would. This uses only the ranking, so it doesn't depend on the base rate.

In [11]:
policies = pipeline.operating_policies(labels, final_preds, thresholds)
print(same_as_saved(policies, "operating_policies.csv"))
policies.round(3)

OK: identical to results/operating_policies.csv


,model,split,policy,flag_rate,precision,recall,lift,threshold
0,route_history,calib,daily top 10%,0.100,0.236,0.151,1.506,NaN
1,route_history,calib,frozen thr for P>=0.4 (from Sep-Oct),0.000,0.667,0.000,4.248,0.434
2,route_history,test,daily top 10%,0.101,0.366,0.143,1.418,NaN
3,route_history,test,frozen thr for P>=0.4 (from Sep-Oct),0.024,0.423,0.039,1.639,0.434
4,logreg_cal,calib,daily top 10%,0.100,0.310,0.198,1.973,NaN
5,logreg_cal,calib,frozen thr for P>=0.4 (from Sep-Oct),0.027,0.400,0.069,2.549,0.363
6,logreg_cal,test,daily top 10%,0.101,0.446,0.174,1.727,NaN
7,logreg_cal,test,frozen thr for P>=0.4 (from Sep-Oct),0.072,0.509,0.142,1.971,0.363
8,xgboost_cal,calib,daily top 10%,0.100,0.301,0.193,1.920,NaN
9,xgboost_cal,calib,frozen thr for P>=0.4 (from Sep-Oct),0.006,0.400,0.014,2.549,0.602


* **The frozen threshold doesn't transfer.** Precision depends on the base rate, so a threshold that barely reaches 40% precision in calm autumn behaves differently in December. XGBoost's frozen threshold flags only **0.6%** of test flights (recall 1.3%).
* **The daily top-10% policy is stable.** It reaches **44% precision and 1.7× lift** over random flagging on test, for both models. This is the policy carried into the Phase 2 rescheduling what-if analysis.

## 8. Feature-family ablation (loaded from `results/ablation.csv`)

The same CV-chosen XGBoost configuration is refit on nested feature sets and scored both on the rolling CV folds and on test.

In [12]:
display(pd.read_csv(R / "ablation.csv"))
figs["ablation_test"]

,features,n_features,cv_pr_auc_mean,cv_pr_auc_std,test_pr_auc,test_roc_auc
0,schedule + ids,13,0.4398,0.0769,0.3447,0.6182
1,+ congestion,20,0.4531,0.0893,0.3451,0.6183
2,+ history_entity,38,0.4498,0.0791,0.3956,0.6542
3,+ history_day,45,0.4637,0.0733,0.3922,0.6637
4,all minus history_day,38,0.4498,0.0791,0.3956,0.6542
5,history only,25,0.4276,0.0739,0.3558,0.6416


<Figure size 550x248 with 1 Axes>

* **Schedule + IDs** already reach a test PR-AUC of 0.345 (time of day is the backbone).
* **+ congestion** adds about **0** on test. The EDA correlation was mostly the evening peak again.
* **+ entity history** gives the largest test gain (**+0.05 PR-AUC**): route, carrier and aircraft propensities.
* **+ day-level history** helps in CV (+0.014; bursty summer storms) but **slightly hurts on test**. That's consistent with the date-identifier concern from notebook 02.
* History alone (0.356) is weaker than schedule + history, so the families complement each other.

## 9. Feature importance

### 9.1 Grouped permutation importance (Sep–Oct, not test)

Each feature family is shuffled **together** with one permutation, so correlated features within a family can't stand in for each other. The drop in PR-AUC is averaged over 5 repeats. This is the primary importance measure.

In [13]:
display(pd.read_csv(R / "permutation_importance_groups.csv"))
figs["perm_importance_groups"]

,group,pr_auc_drop,std
0,categorical,0.0096,0.0010
1,schedule,0.0484,0.0011
2,congestion,0.0083,0.0007
3,history_entity,0.0121,0.0011
4,history_day,0.0159,0.0011


<Figure size 550x220 with 1 Axes>

### 9.2 XGBoost gain (top 20)

In [14]:
figs["xgb_gain_top20"]

<Figure size 550x640 with 1 Axes>

### 9.3 Logistic regression coefficients (largest |β|, standardised features)

In [15]:
pd.read_csv(R / "logreg_coefficients.csv", index_col=0).head(20)

,coef
num__sched_origin_day,0.2901
num__hist_rate_carrier_7d,0.2822
num__dep_hour,0.2294
num__hist_rate_carrier_flight_28d,0.2289
cat__carrier_EV,0.1971
cat__dest_BOS,-0.1963
num__hist_meandelay_origin_1d,0.1682
num__hist_rate_carrier_28d,-0.1651
num__hist_rate_network_7d,-0.1576
num__hist_rate_carrier_origin_dest_28d,0.1553


How to read the three views:
* **Permutation (by group):** the **schedule** family (time of day) matters most by a wide margin (PR-AUC drop 0.048), followed by day-level history (0.016), entity history (0.012), IDs and congestion (about 0.01 each).
* **XGBoost gain:** top features are `dep_hour`, flight-number history (`hist_rate_carrier_flight_28d`), origin × hour history and `dep_tod_sin`. Time of day plus fine-grained entity history, matching the correlations in notebook 02.
* **LogReg coefficients must be read with care.** They are *conditional* on all other features, so correlated features share or split credit:
  * `sched_origin_day` has the largest coefficient (+0.29), yet the congestion family adds nothing in the ablation. Once time of day and history are in the model, its effect is a correction term, not a standalone driver.
  * `hist_rate_carrier_7d` (+0.28) and `hist_rate_carrier_28d` (−0.17) have **opposite signs**. Together they encode a *trend*: a carrier whose last week is worse than its last month is currently deteriorating.
  * This is why grouped permutation, rather than coefficients or gain, is the primary importance measure.

## 10. Findings

1. **Both models beat the baselines.** Test PR-AUC is 0.39–0.41 (base rate 0.26), and at precision ≥ 0.40 they recover 43–48% of late flights, versus 9% for the route heuristic.
2. **XGBoost ≈ LogReg.** The day-block bootstrap CI for the difference includes 0. The day-ahead signal is weak and mostly additive, and both searches chose strong regularisation.
3. **Time of day and entity history carry the signal.** Congestion adds nothing beyond time of day, and day-level history doesn't transfer to test.
4. **Calibration is a property of the model *and* the data distribution.** Recalibration on calm Sep–Oct breaks in December, and raw log-loss models were better calibrated.
5. **Daily-capacity flagging is the robust operating policy**: 44% precision, 1.7× lift. A precision threshold frozen on one season doesn't transfer to the next.

## 11. Limitations
* **One year, three origins.** Seasonality and airport effects are each observed only once.
* **No weather data.** This is the main missing driver of day-to-day delays, which caps how well a day-ahead model can do.
* **Only NYC departures.** Aircraft rotation is only partly visible (about 1.3 NYC departures per aircraft per day).
* **`tailnum` is assumed known at D-1** (realistic for published routings, but an assumption).
* **Single selection run.** The LogReg calibrator choice came down to a near-tie that didn't transfer. Selection noise like this would shrink with a longer calibration window.